# رقيب · Road-Risk Prediction — Notebook (v2)
Predict road **accident severity** from **10 intuitive road/environment features**.
Dataset: `RTA Dataset.csv`. Model: HistGradientBoosting (class-weighted).


## 1. Setup


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, joblib, json
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
CSV='RTA Dataset.csv'


## 2. Load, derive Hour, select intuitive features


In [ ]:
df=pd.read_csv(CSV).replace(['na','Unknown','unknown','nan','NaN',''], np.nan)
df['Hour']=pd.to_datetime(df['Time'], errors='coerce').dt.hour
FEATURES=['Road_surface_type','Road_surface_conditions','Light_conditions','Weather_conditions',
          'Road_allignment','Lanes_or_Medians','Types_of_Junction',
          'Number_of_vehicles_involved','Hour','Day_of_week']
y=df['Accident_severity'].astype(str); X=df[FEATURES]
y.value_counts(normalize=True).round(3)


## 3. Preprocess + model


In [ ]:
NUM=['Number_of_vehicles_involved','Hour']; CAT=[c for c in FEATURES if c not in NUM]
pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),NUM),
  ('cat',Pipeline([('i',SimpleImputer(strategy='most_frequent')),
                   ('o',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),CAT)])
model=Pipeline([('pre',pre),('clf',HistGradientBoostingClassifier(random_state=42,max_iter=300,class_weight='balanced'))])


## 4. Evaluate on held-out test


In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
model.fit(Xtr,ytr); pred=model.predict(Xte)
print('balanced_accuracy=%.3f'%balanced_accuracy_score(yte,pred))
print('macro_F1=%.3f'%f1_score(yte,pred,average='macro'))
print(classification_report(yte,pred))
confusion_matrix(yte,pred,labels=sorted(y.unique()))


## 5. Refit on all data & save


In [ ]:
model.fit(X,y); joblib.dump(model,'road_risk_model.joblib')
schema={'target':'Accident_severity','classes':sorted(y.unique()),'numeric':NUM,
        'categorical':{c:sorted(X[c].dropna().unique().tolist()) for c in CAT}}
json.dump(schema,open('feature_schema.json','w'),ensure_ascii=False,indent=2)
print('saved')


## 6. Intuitive rule-based index + sample


In [ ]:
from risk_rules import road_risk_index
bad={'Road_surface_type':'Earth roads','Road_surface_conditions':'Flood over 3cm. deep',
     'Light_conditions':'Darkness - no lighting','Weather_conditions':'Raining',
     'Road_allignment':'Steep grade downward with mountainous terrain',
     'Lanes_or_Medians':'Undivided Two way','Types_of_Junction':'Y Shape',
     'Number_of_vehicles_involved':3,'Hour':2,'Day_of_week':'Saturday'}
road_risk_index(bad)


---
Serve with `uvicorn app:app --port 8000` (see README). API returns both the intuitive index and the ML prediction.
